# COMP - 2040: Final Project  
**Name:** Troy Dela Rosa  
**SID#** 0213352  
**Date** April 07, 2026  
**Instructor:** Chris Mac  

## Downtown Decline: An Analysis of Business License Activity in Winnipeg (2023–2025)  
**Dataset:** City of Winnipeg Open Data – Business Licenses (exported April 4, 2026)  
**Dataset size:** 7,313 records | 13 columns

### Problem Statement

Downtown Winnipeg has long faced challenges with commercial vacancy, foot traffic decline, and economic displacement — trends that continued in the post-pandemic period. This project uses City of Winnipeg business license data to investigate whether license activity patterns reflect a measurable decline, and which business types and neighbourhoods are most affected.


**Core question:** *Is downtown Winnipeg experiencing a decline in active business licenses, and what patterns — by industry, neighbourhood, and time?*

### Analytical Decisions Made Upfront

- **"Closure" is defined** as any license with Status of `Closed (L)`, `Ceased Operation`, or `Cancelled`. The dataset does not contain an explicit closure field — this is a standard proxy definition used in business license analytics.
- **"Downtown"** is defined using the `Community Characterization Area` column value `Downtown`, which the City of Winnipeg assigns to 14 neighbourhoods including Exchange District, South Portage, West Broadway, and Spence.
- **2021 and 2022 are excluded from trend analysis** because these years represent the height of COVID-19 disruption and post-lockdown re-licensing surges, which distort normal business activity patterns. Including them would conflate pandemic-era anomalies with structural trends. The analysis focuses on 2023–2025 as a cleaner, post-stabilization window.
- **2026 data is also excluded** as the year is incomplete (only 1 downtown record as of export date).

### Week 12

#### Setup and Import

In [5]:
import pandas as pd

df = pd.read_csv('data/Business_Licenses_20260404.csv')

# Parse dates in 'Issue Date' and 'Expiry Date' columns
df['Issue Date'] = pd.to_datetime(df['Issue Date'])
df['Expiry Date'] = pd.to_datetime(df['Expiry Date'])

print(f'Dataset shape: {df.shape}')
df.head(10)

Dataset shape: (7313, 13)


C:\Users\troyd\AppData\Local\Temp\ipykernel_90452\16826002.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Issue Date'] = pd.to_datetime(df['Issue Date'])
C:\Users\troyd\AppData\Local\Temp\ipykernel_90452\16826002.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Expiry Date'] = pd.to_datetime(df['Expiry Date'])


,Folder Type,Folder Description,Subdescription,Trade Name,Address,Issue Date,Expiry Date,Status,Neighbourhood Number,Neighbourhood Name,Electoral Ward,Community Characterization Area,Location
0,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2023-05-10,2024-05-31,Issued,3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
1,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2022-05-10,2023-05-31,Closed (L),3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
2,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2023-05-10,2024-05-31,Issued,3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
3,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2022-05-10,2023-05-31,Closed (L),3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
4,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2021-05-17,2022-05-31,Closed (L),3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
5,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2022-05-10,2023-05-31,Closed (L),3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
6,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2023-05-10,2024-05-31,Issued,3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
7,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2022-05-10,2023-05-31,Closed (L),3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
8,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2023-05-10,2024-05-31,Issued,3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)
9,LP,License - Personal Sales,Bicycle Dealer,Natural Cycle,91 Albert ST,2022-05-10,2023-05-31,Closed (L),3.17,EXCHANGE DISTRICT,Point Douglas,Downtown,POINT (-97.13994322809528 49.8975051398304)


In [6]:
df.tail

<bound method NDFrame.tail of      Folder Type        Folder Description         Subdescription  \
0             LP  License - Personal Sales         Bicycle Dealer   
1             LP  License - Personal Sales         Bicycle Dealer   
2             LP  License - Personal Sales         Bicycle Dealer   
3             LP  License - Personal Sales         Bicycle Dealer   
4             LP  License - Personal Sales         Bicycle Dealer   
...          ...                       ...                    ...   
7308          LP  License - Personal Sales      Used Goods Dealer   
7309          LP  License - Personal Sales      Used Goods Dealer   
7310          LP  License - Personal Sales      Used Goods Dealer   
7311          LP  License - Personal Sales  Precious Metal Dealer   
7312          LP  License - Personal Sales  Precious Metal Dealer   

                 Trade Name             Address Issue Date Expiry Date  \
0             Natural Cycle        91 Albert ST 2023-05-10  2024-05

### Initial Observations - `df.head` and `df.tail`

1. The dataset contains 7,313 records and 13 columns, indicating a moderately sized dataset with sufficient data for meaningful analysis.  
2. The "Status" field includes values such as "Issued" and "Closed (L)", suggesting a mix of active and inactive business licenses, which can be used to analyze business survival and turnover.  
3. The Issue Date and Expiry Date columns show consistent yearly patterns (e.g., many licenses expiring on May 31), indicating that licenses are likely issued on a fixed annual cycle rather than continuously.  
4. There are repeated entries for the same business (e.g., "Natural Cycle"), implying that businesses renew licenses over time. This allows for longitudinal analysis of business continuity and retention.  
5. Geographic fields such as Neighbourhood Name (e.g., "EXCHANGE DISTRICT") and Electoral Ward (e.g., "Point Douglas") suggest the dataset can support spatial analysis of business distribution and concentration.  
6. Some fields, such as Address or Trade Name, contain missing values (e.g., "(none)"), indicating data quality issues that may need to be handled during cleaning.  
7. The Location field stores coordinates in POINT format, which can be converted into latitude and longitude for mapping and geospatial visualization.  
8. The dataset includes multiple business categories (e.g., Bicycle Dealer, Used Goods Dealer, Precious Metal Dealer), enabling segmentation analysis by industry type.